# 12 · itertools and functools

*Standard library toolbox* · 9 exercises

Generate permutations, combinations and products for safe brute force.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-12) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-12)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import math
import operator
from functools import reduce
from itertools import accumulate, chain, combinations, groupby, islice, count, permutations, product

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 12.1 · permutations  `Core`

**Task.** Write `numbers_from_digits(digits)` returning every distinct number formed by using all the digits once, sorted. Use `itertools.permutations`.

**Example.** `numbers_from_digits([1, 2, 1])` -> `[112, 121, 211]`

**Big O.** O(N! * N). Practical up to N = 8 or so.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-1)

In [ ]:
# 12.1 permutations
def numbers_from_digits(digits):
    raise NotImplementedError

In [ ]:
def test_ex_12_1():
    assert numbers_from_digits([1, 2, 1]) == [112, 121, 211]


test_ex_12_1()
print('12.1 passed')

### 12.2 · combinations  `Core`

**Task.** Write `k_subsets_with_sum(nums, k, target)` returning all k-element combinations (as tuples, in generation order) that sum to target.

**Example.** `k_subsets_with_sum([1, 2, 3, 4, 5], 2, 6)` -> `[(1, 5), (2, 4)]`

**Big O.** O(C(N, k) * k).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-2)

In [ ]:
# 12.2 combinations
def k_subsets_with_sum(nums, k, target):
    raise NotImplementedError

In [ ]:
def test_ex_12_2():
    assert k_subsets_with_sum([1, 2, 3, 4, 5], 2, 6) == [(1, 5), (2, 4)]


test_ex_12_2()
print('12.2 passed')

### 12.3 · product  `Core`

**Task.** Write `dice_sum_counts(n)` returning a dict of total to number of ways when rolling n six-sided dice, using `product(range(1, 7), repeat=n)`.

**Example.** `dice_sum_counts(2)[7]` -> `6`; `sum(dice_sum_counts(3).values())` -> `216`

**Big O.** O(6^N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-3)

In [ ]:
# 12.3 product
def dice_sum_counts(n):
    raise NotImplementedError

In [ ]:
def test_ex_12_3():
    assert dice_sum_counts(2)[7] == 6
    assert sum(dice_sum_counts(3).values()) == 216


test_ex_12_3()
print('12.3 passed')

### 12.4 · accumulate  `Core`

**Task.** Write `prefix(arr)` with `accumulate` (starting from 0, so the result has N + 1 values), and `running_best(arr)` with `accumulate(arr, max)`.

**Example.** `prefix([3, 1, 4])` -> `[0, 3, 4, 8]`; `running_best([2, 1, 5, 3])` -> `[2, 2, 5, 5]`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-4)

In [ ]:
# 12.4 accumulate
def prefix(arr):
    raise NotImplementedError


def running_best(arr):
    raise NotImplementedError

In [ ]:
def test_ex_12_4():
    assert prefix([3, 1, 4]) == [0, 3, 4, 8]
    assert running_best([2, 1, 5, 3]) == [2, 2, 5, 5]


test_ex_12_4()
print('12.4 passed')

### 12.5 · groupby  `Core`

**Task.** Write `runs(s)` returning `(char, length)` for each run of equal characters using `groupby`. Then explain why groupby on unsorted data does not group all equal items together.

**Example.** `runs("aaabccdddd")` -> `[('a', 3), ('b', 1), ('c', 2), ('d', 4)]`

**Big O.** O(N).

**Note.** `groupby` only merges neighbours. Sort first if you want one group per key.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-5)

In [ ]:
# 12.5 groupby
def runs(s):
    raise NotImplementedError

In [ ]:
def test_ex_12_5():
    assert runs('aaabccdddd') == [('a', 3), ('b', 1), ('c', 2), ('d', 4)]
    assert runs('') == []


test_ex_12_5()
print('12.5 passed')

### 12.6 · chain  `Core`

**Task.** Write `merge_all(lists)` that concatenates any number of lists with `chain.from_iterable`, and `interleave(a, b)` alternating items of two equal-length lists.

**Example.** `merge_all([[1], [2, 3], []])` -> `[1, 2, 3]`; `interleave([1, 3], [2, 4])` -> `[1, 2, 3, 4]`

**Big O.** O(total items).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-6)

In [ ]:
# 12.6 chain
def merge_all(lists):
    raise NotImplementedError


def interleave(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_12_6():
    assert merge_all([[1], [2, 3], []]) == [1, 2, 3]
    assert interleave([1, 3], [2, 4]) == [1, 2, 3, 4]


test_ex_12_6()
print('12.6 passed')

### 12.7 · reduce  `Core`

**Task.** Write `product_of(nums)` with `reduce(operator.mul, nums, 1)` and `gcd_of(nums)` with `reduce(math.gcd, nums)`.

**Example.** `product_of([2, 3, 7])` -> `42`; `gcd_of([12, 18, 30])` -> `6`

**Big O.** O(N) operations. `math.prod` and `math.gcd(*nums)` (3.9+) do the same.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-7)

In [ ]:
# 12.7 reduce
def product_of(nums):
    raise NotImplementedError


def gcd_of(nums):
    raise NotImplementedError

In [ ]:
def test_ex_12_7():
    assert product_of([2, 3, 7]) == 42
    assert product_of([]) == 1
    assert gcd_of([12, 18, 30]) == 6


test_ex_12_7()
print('12.7 passed')

### 12.8 · Lazy sequences with count and islice  `Practice`

**Task.** Write `first_n_primes(n)` that walks `count(2)`, filters primes with a simple test, and takes n with `islice`.

**Example.** `first_n_primes(6)` -> `[2, 3, 5, 7, 11, 13]`

**Big O.** About O(n * sqrt(p_n)) for the trial-division test.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-8)

In [ ]:
# 12.8 Lazy sequences with count and islice
def first_n_primes(n):
    raise NotImplementedError

In [ ]:
def test_ex_12_8():
    assert first_n_primes(6) == [2, 3, 5, 7, 11, 13]


test_ex_12_8()
print('12.8 passed')

### 12.9 · Brute force every subset  `Practice`

**Task.** Write `closest_subset_sum(nums, target)` returning the subset sum closest to target (prefer the smaller sum on ties), trying every size with `combinations`.

**Example.** `closest_subset_sum([8, 3, 5, 11], 15)` -> `14`; `closest_subset_sum([8, 3, 5, 11], 16)` -> `16`

**Big O.** O(2^N * N). Fine for N up to about 20.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-12-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-12-9)

In [ ]:
# 12.9 Brute force every subset
def closest_subset_sum(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_12_9():
    assert closest_subset_sum([8, 3, 5, 11], 15) == 14
    assert closest_subset_sum([8, 3, 5, 11], 16) == 16


test_ex_12_9()
print('12.9 passed')